# [LAB-09] 4. 다중선형회귀 - 연습문제

## 준비작업

### 라이브러리 참조

In [1]:
import numpy as np
from jussam import load_data
from helpers import *

📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/


## 📚 신입 분석가의 냉방부하 설계 보고서

### 당신은 에너지 컨설팅 회사에 갓 입사한 데이터 분석가입니다.

건축 설계팀이 시뮬레이션으로 만들어 낸 768개 건물 형태의 설계 제원(건물조밀도, 표면적, 벽면적, 지붕면적, 높이, 유리창면적, 유리창면적분포)과 그때의 냉방부하가 enb 데이터 셋에 담겨 있습니다.

설계 팀장은 "에어컨 용량을 줄이려면 설계 단계에서 무엇을 손봐야 하는지 딱 하나만 짚어 달라"고 요구하지만, 제원들끼리 서로 강하게 얽혀 있어서 그냥 모든 변수를 회귀식에 넣으면 계수를 믿을 수 없습니다.

데이터를 정제하고 서로 겹치는 변수를 걷어낸 뒤, 냉방부하를 좌우하는 요인이 무엇인지 회귀분석으로 확인하세요.

### 💻 코드 작성

#### 데이터 불러오기

In [2]:
origin = load_data('enb')
origin.head()

📚 이 데이터 세트는 토목/구조 엔지니어인 Angeliki Xifara(angxifara@gmail.com)가 생성했으며, 영국 옥스퍼드 대학교 산업 및 응용 수학 센터의 Athanasios Tsanas(tsanasthanasis@gmail.com)가 처리했습니다.

본 연구에서는 Ecotect를 이용하여 12가지 서로 다른 건물 형태를 시뮬레이션하고 에너지 분석을 수행했습니다.

건물들은 여러 변수에서 차이를 보입니다.

이러한 특성들을 함수로 하여 다양한 설정을 시뮬레이션함으로써 총 768가지 건물 형태를 얻었습니다.

데이터셋은 768개의 샘플과 8개의 특징으로 구성되며, 두 개의 실수 값을 갖는 응답값을 예측하는 것을 목표로 합니다.

(출처: https://www.kaggle.com/datasets/elikplim/eergy-efficiency-dataset)

> 모든 데이터는 연속형 변수 입니다.


,건물조밀도,표면적,벽면적,지붕면적,높이,유리창면적,유리창면적분포,냉방부하
0,0.980,514.500,294.000,110.250,7.000,0.000,0,21.330
1,0.980,514.500,294.000,110.250,7.000,0.000,0,21.330
2,0.980,514.500,294.000,110.250,7.000,0.000,0,21.330
3,0.980,514.500,294.000,110.250,7.000,0.000,0,21.330
4,0.900,563.500,318.500,122.500,7.000,0.000,0,28.280


#### 결측치 확인

In [3]:
my_qtcheck.check_missing_values(origin)

,Missing Count,Missing Ratio (%)
건물조밀도,0,0.000
표면적,0,0.000
벽면적,0,0.000
지붕면적,0,0.000
높이,0,0.000
유리창면적,0,0.000
유리창면적분포,0,0.000
냉방부하,0,0.000


#### 중복 데이터 제거

In [4]:
df1 = my_qtcheck.check_duplicates(origin)

print(f"중복 제거 후 관측치 개수: {len(df1)}")

df1.head()

중복된 행의 수 : 16
중복된 행이 제거되었습니다.
중복 제거 후 관측치 개수: 752


,건물조밀도,표면적,벽면적,지붕면적,높이,유리창면적,유리창면적분포,냉방부하
0,0.980,514.500,294.000,110.250,7.000,0.000,0,21.330
4,0.900,563.500,318.500,122.500,7.000,0.000,0,28.280
5,0.900,563.500,318.500,122.500,7.000,0.000,0,25.380
6,0.900,563.500,318.500,122.500,7.000,0.000,0,25.160
7,0.900,563.500,318.500,122.500,7.000,0.000,0,29.600


#### 기초 통계량 확인

In [5]:
my_qtcheck.numerical_summary(df1).T

,건물조밀도,표면적,벽면적,지붕면적,높이,유리창면적,유리창면적분포,냉방부하
count,752.000,752.000,752.000,752.000,752.000,752.000,752.000,752.000
mean,0.765,671.111,318.337,176.387,5.264,0.235,2.846,24.705
std,0.105,87.291,43.550,45.031,1.751,0.133,1.536,9.553
min,0.620,514.500,245.000,110.250,3.500,0.000,0.000,10.900
25%,0.690,606.375,294.000,147.000,3.500,0.100,2.000,15.620
50%,0.760,661.500,318.500,147.000,7.000,0.250,3.000,22.725
75%,0.830,735.000,343.000,220.500,7.000,0.400,4.000,33.233
max,0.980,808.500,416.500,220.500,7.000,0.400,5.000,48.030
rel_diff,0.006,0.015,0.001,0.200,0.248,0.061,0.051,0.087
rdiff_flag,similar,similar,similar,diff,diff,similar,similar,similar


#### 로그 변환

In [6]:
df2 = df1.copy()
df2['벽면적'] = np.log1p(df2['벽면적'])
df2.head()

,건물조밀도,표면적,벽면적,지붕면적,높이,유리창면적,유리창면적분포,냉방부하
0,0.980,514.500,5.687,110.250,7.000,0.000,0,21.330
4,0.900,563.500,5.767,122.500,7.000,0.000,0,28.280
5,0.900,563.500,5.767,122.500,7.000,0.000,0,25.380
6,0.900,563.500,5.767,122.500,7.000,0.000,0,25.160
7,0.900,563.500,5.767,122.500,7.000,0.000,0,29.600


#### 다중 공선성 제거

In [7]:
xnames = ['건물조밀도', '표면적', '벽면적', '지붕면적', '높이', '유리창면적', '유리창면적분포']

# 변수 제거 전의 VIF
display(my_stats.compute_vif(df2[xnames]))

# VIF가 가장 큰 변수부터 하나씩 반복 제거
df3 = my_prep.reduce_vif(df2, columns=xnames)
df3.head()

,VIF
표면적,1871.738
지붕면적,833.052
벽면적,255.767
건물조밀도,203.748
높이,57.007
유리창면적,1.045
유리창면적분포,1.045


[1단계] 표면적 제거 (VIF = 1871.7)
[2단계] 지붕면적 제거 (VIF = 116.9)

완료! 남은변수 : ['건물조밀도', '높이', '벽면적', '유리창면적', '유리창면적분포']
최대 VIF = 9.71


,건물조밀도,벽면적,높이,유리창면적,유리창면적분포,냉방부하
0,0.980,5.687,7.000,0.000,0,21.330
4,0.900,5.767,7.000,0.000,0,28.280
5,0.900,5.767,7.000,0.000,0,25.380
6,0.900,5.767,7.000,0.000,0,25.160
7,0.900,5.767,7.000,0.000,0,29.600


#### 1차 회귀분석 및 가정 확인

In [8]:
fit1 = my_ols.auto_ols(df3, y='냉방부하', log_x=['벽면적'], test=True)

#### -> 모형 적합도

,종속변수,독립변수,B,표준오차,표준오차(HC3),베타,t,t(HC3),유의확률,유의확률(HC3),공차,VIF
0,냉방부하,높이,5.666,0.211,0.281,1.039,26.848,20.179,0.000,0.000,0.103,9.707
1,냉방부하,건물조밀도,-18.881,3.418,3.927,-0.207,-5.525,-4.808,0.000,0.000,0.110,9.112
2,냉방부하,유리창면적,14.494,0.914,0.925,0.201,15.849,15.671,0.000,0.000,0.957,1.045
3,냉방부하,벽면적,6.571,1.572,1.578,0.092,4.181,4.163,0.000,0.000,0.316,3.166
4,냉방부하,유리창면적분포,0.020,0.079,0.080,0.003,0.255,0.250,0.799,0.803,0.957,1.045


**Note. n = 752. F(5, 746) = 1148.22, p < 0.001, R^2 = 0.885, Adj.R^2 = 0.884, Durbin-Watson = 1.052**

냉방부하를 종속변수로, 건물조밀도, log(벽면적), 높이, 유리창면적, 유리창면적분포(을)를 독립변수로한 다중선형회귀분석 결과, 모형은 통계적으로 유의하였다..

> F(5, 746) = 1148.22, p < 0.001, R^2 = 0.885.

즉, 건물조밀도, log(벽면적), 높이, 유리창면적, 유리창면적분포는 냉방부하의 약 88.5%를 설명하는 것으로 나타났다.

---

#### -> 회귀모형 가정 검정

...

루트MSE = 3.25 , 구간 규칙 판정: 정규성 위배


##### 3) 등분산성 검정

,LM statistic,LM p-value,F statistic,F p-value,homoscedasticity,result
Breusch-Pagan,181.827,0.000,47.579,0.000,False,대립가설 채택 -> 등분산 아님


##### 4) 독립성 검정

,statistic,independence,result
Durbin-Watson,1.052,False,독립성 위반 (양(+)의 자기상관)


#### 2차 회귀분석 (유의하지 않은 변수 제거)

In [9]:
df4 = df3.drop(columns=['유리창면적분포'])
fit2 = my_ols.auto_ols(df4, y='냉방부하', log_x=['벽면적'], test=False)

#### -> 모형 적합도

,종속변수,독립변수,B,표준오차,표준오차(HC3),베타,t,t(HC3),유의확률,유의확률(HC3),공차,VIF
0,냉방부하,높이,5.665,0.211,0.280,1.038,26.865,20.205,0.000,0.000,0.103,9.704
1,냉방부하,건물조밀도,-18.864,3.415,3.920,-0.207,-5.524,-4.812,0.000,0.000,0.110,9.109
2,냉방부하,유리창면적,14.542,0.894,0.910,0.202,16.260,15.980,0.000,0.000,0.999,1.001
3,냉방부하,벽면적,6.576,1.571,1.576,0.092,4.187,4.173,0.000,0.000,0.316,3.166


**Note. n = 752. F(4, 747) = 1437.06, p < 0.001, R^2 = 0.885, Adj.R^2 = 0.884, Durbin-Watson = 1.052**

냉방부하를 종속변수로, 건물조밀도, log(벽면적), 높이, 유리창면적(을)를 독립변수로한 다중선형회귀분석 결과, 모형은 통계적으로 유의하였다..

> F(4, 747) = 1437.06, p < 0.001, R^2 = 0.885.

즉, 건물조밀도, log(벽면적), 높이, 유리창면적는 냉방부하의 약 88.5%를 설명하는 것으로 나타났다.

### 문제 풀이

#### 1. 시뮬레이션 데이터에는 제원이 완전히 똑같은 설계안이 여러 번 들어가 있습니다. 같은 내용이 반복된 행을 정리하고 나면 분석에 쓰이는 관측치는 몇 개가 되나요?

- **정답**: `752`
- **복습 개념**: 회귀분석에 들어가기 전의 데이터 품질 점검(결측치·중복치)입니다. 똑같은 행이 여러 번 들어가 있으면 그 설계안에만 가중치가 실려 계수와 표준오차가 왜곡되므로 분석 전에 걷어내야 합니다.
- **풀이 접근**: 먼저 결측치가 있는지 확인하고(이 데이터는 8개 열 모두 결측 0건입니다), 이어서 모든 열의 값이 동일한 행을 찾아 제거한 뒤 남은 행 수를 셉니다.
- **근거(계산 결과)**: 원본은 768행인데 완전히 동일한 행이 16개 발견되어 제거되었습니다. 768 − 16 = **752행**이며, 위 코드의 출력에서 `중복된 행의 수: 16`, `중복 제거 후 관측치 개수: 752` 로 확인됩니다.
- **자주 하는 실수**: 데이터 설명에 적힌 768을 그대로 답하기 쉽습니다. 이 752는 뒤에 나오는 회귀 결과의 `n = 752`, 그리고 F(3, 748)의 잔차 자유도 748(= 752 − 3 − 1)과 그대로 이어지니 기억해 두세요.

#### 2. 기초 통계량을 보면 7개 제원 중 단 하나만 분포가 오른쪽으로 치우쳐 있어 로그 변환 대상으로 진단됩니다. 그 변수는 무엇인가요?

- **정답**: `벽면적`
- **복습 개념**: 왜도(skewness)를 근거로 한 변수 변환 판단입니다. 회귀분석은 잔차의 정규성을 가정하는데, 한쪽으로 꼬리가 긴 변수를 그대로 넣으면 큰 값들이 잔차를 끌고 다닙니다. 로그를 씌우면 큰 값이 더 많이 압축되어 분포가 대칭에 가까워집니다.
- **풀이 접근**: 기초 통계량 표에서 각 변수의 왜도 값과 그 해석을 확인합니다. 왜도가 양수이고 그 크기가 뚜렷하면 오른쪽 꼬리(right tail)로, 로그 변환 후보가 됩니다.
- **근거(계산 결과)**: `skew` 행을 보면 벽면적만 0.548로 `right tail` 판정을 받았고 `log_need` 행에도 유일하게 `log1p` 가 찍혀 있습니다. 나머지는 건물조밀도 0.485, 냉방부하 0.373 등으로 모두 `symmetric` 입니다. 그래서 다음 단계에서 벽면적에만 `np.log1p()` 를 적용합니다.
- **헷갈리기 쉬운 점**: 로그 변환을 하면 회귀계수의 해석도 바뀝니다. 원래 값이 아니라 log(벽면적)이 1 늘어날 때 냉방부하가 얼마나 변하는지를 뜻하게 되므로, 결과를 보고할 때는 반드시 '로그를 씌운 벽면적'이라고 밝혀야 합니다.

#### 3. 건물의 제원들은 서로 강하게 얽혀 있습니다. 로그 변환을 수행한 후 분산팽창지수를 기준으로 겹치는 변수를 하나씩 걷어낼 때, 가장 먼저 제거되는 변수는 무엇인가요?

- **정답**: `표면적`
- **복습 개념**: 다중공선성(multicollinearity)과 VIF입니다. VIF는 "그 변수를 나머지 독립변수들로 얼마나 잘 설명할 수 있는가"를 나타내며, 보통 10을 넘으면 공선성이 심하다고 봅니다. 공선성이 심하면 계수의 표준오차가 부풀어 부호가 뒤집히거나 유의성이 사라집니다.
- **풀이 접근**: 독립변수들만 모아 VIF를 계산하고, 값이 가장 큰 변수를 하나 제거한 뒤 다시 계산하기를 모든 VIF가 기준값 아래로 내려갈 때까지 반복합니다. 한 번에 전부 지우지 않고 하나씩 지우는 이유는, 하나를 빼면 나머지 변수들의 VIF가 크게 떨어지기 때문입니다.
- **근거(계산 결과)**: 첫 계산에서 표면적의 VIF가 **1871.7** 로 압도적으로 큽니다(지붕면적 833.1, 벽면적 255.8, 건물조밀도 203.7, 높이 57.0). 실제로 건물의 표면적은 벽면적과 지붕면적의 합에 가깝기 때문에 다른 변수들로 거의 완벽하게 재현됩니다. 제거 과정 출력도 `[1단계] 표면적 제거 → [2단계] 지붕면적 제거 → [3단계] 높이 제거` 순으로 진행되어, 최종적으로 건물조밀도·벽면적·유리창면적·유리창면적분포가 남고 최대 VIF는 4.42로 안정됩니다.
- **실무 포인트**: 표면적이 냉방부하와 무관해서 빠진 것이 아닙니다. 다른 변수들이 이미 같은 정보를 담고 있어 중복이라 빠진 것입니다. "제거 = 중요하지 않음"이 아니라 "제거 = 남은 변수로 대체 가능함"으로 읽어야 합니다.

#### 4. 공선성을 정리한 4개 변수로 1차 회귀분석을 돌리면, 유의수준 5% 기준으로 딱 하나의 변수만 냉방부하와 무관하다고 판정됩니다. 그 변수는 무엇인가요?

- **정답**: `유리창면적분포`
- **복습 개념**: 회귀계수의 개별 유의성 검정입니다. 각 독립변수마다 "이 변수의 모집단 계수는 0이다"라는 귀무가설을 t 검정으로 판단하며, 유의확률이 0.05보다 크면 0이라는 가설을 기각하지 못합니다.
- **풀이 접근**: 회귀 결과표에서 각 독립변수의 유의확률 열을 읽고 0.05와 비교합니다. 등분산 가정이 깨진 경우에는 HC3(로버스트) 표준오차 기준 유의확률을 함께 봅니다.
- **근거(계산 결과)**: 유리창면적분포의 유의확률은 **0.869** (HC3 기준도 0.869)로 0.05보다 훨씬 큽니다. 계수도 −0.018로 사실상 0이고 t = −0.165에 불과합니다. 나머지 세 변수(건물조밀도, 벽면적, 유리창면적)는 모두 p < 0.001 입니다. 유리창의 '면적'은 냉방부하를 늘리지만, 그 유리창을 건물의 어느 방향에 어떻게 '배치'했는지는 이 모형에서 냉방부하를 설명하지 못한다는 뜻입니다. 그래서 2차 분석에서 이 변수를 빼고 다시 적합합니다.
- **함께 생각해 볼 점**: 유리창면적분포는 0~5의 정수 코드로, 크기가 아니라 배치 유형을 나타내는 명목형에 가까운 값입니다. 이런 변수를 연속형처럼 그대로 넣으면 "코드가 1 커질수록 부하가 일정하게 변한다"는 이상한 가정을 하게 됩니다. 유의하지 않다는 결론 앞에, 변수를 잘못 넣은 것은 아닌지도 함께 의심해 보세요.

#### 5. 1차 분석에서는 회귀모형의 네 가지 가정(선형성·정규성·등분산성·독립성)을 모두 검정합니다. 이 중 충족된 것으로 판정된 가정은 몇 개인가요?

- **정답**: `0`
- **복습 개념**: 회귀모형의 가정 검정입니다. 선형성은 Ramsey RESET, 정규성은 콜모고로프–스미르노프, 등분산성은 Breusch–Pagan, 독립성은 더빈–왓슨으로 확인합니다. 앞의 세 검정은 유의확률이 0.05보다 크면 가정이 충족되고(귀무가설이 곧 '가정이 맞다'), 더빈–왓슨은 값이 2에 가까울수록 자기상관이 없습니다.
- **풀이 접근**: 가정 검정 출력의 네 표를 차례로 읽어 판정 열이 True(충족)인 것의 개수를 셉니다.
- **근거(계산 결과)**: 선형성 p = 0.013 → 위배, 정규성 p < 0.001 → 위배, 등분산성 F 검정 p < 0.001 → 위배, 더빈–왓슨 = 0.643 (2에서 크게 벗어남) → 독립성 위배. 충족된 가정은 **0개**입니다.
- **실무 포인트**: 이 데이터는 12가지 기본 형태를 조금씩 바꿔 만든 시뮬레이션이라 비슷한 설계안이 줄줄이 붙어 있고, 그래서 잔차가 이웃한 행끼리 닮게 됩니다(더빈–왓슨 0.643). 등분산이 깨졌기 때문에 결과표가 HC3 로버스트 표준오차를 함께 보여준다는 점도 확인하세요. 가정이 깨졌다고 모형이 쓸모없어지는 것은 아니지만, 계수의 유의성을 단정적으로 말할 때는 조심해야 합니다.

#### 6. 유의하지 않은 변수를 뺀 최종 모형이 냉방부하의 변동을 설명하는 정도를, 독립변수 개수를 보정해서 나타낸 값은 얼마인가요? (소수 셋째 자리)

- **정답**: `0.773`
- **복습 개념**: 결정계수(R²)와 수정된 결정계수(Adjusted R²)입니다. R²는 변수를 아무거나 더 넣기만 해도 절대 줄어들지 않기 때문에, 변수 개수에 대한 벌점을 준 Adj. R²로 모형끼리 비교합니다.
- **풀이 접근**: 최종 모형의 적합도 보고를 확인해 R²와 Adj. R²를 읽고, 변수 개수를 보정한 쪽 값을 답합니다.
- **근거(계산 결과)**: 최종 모형은 F(3, 748) = 853.33, p < 0.001, R² = 0.774, **Adj. R² = 0.773** 입니다. 즉 세 변수만으로 냉방부하 변동의 약 77.4%가 설명됩니다. 유리창면적분포를 포함했던 1차 모형도 R²는 똑같이 0.774였는데, 이는 그 변수가 설명력에 아무것도 보태지 못했다는 뜻입니다. 반면 F 통계량은 639.2에서 853.3으로 올라 모형이 더 간결해졌습니다.
- **헷갈리기 쉬운 점**: R²(0.774)와 Adj. R²(0.773)를 혼동하지 마세요. 쓸모없는 변수를 빼면 R²는 그대로여도 Adj. R²는 올라갑니다. 변수를 넣을지 말지 판단할 때 봐야 하는 쪽은 항상 Adj. R² 입니다.

#### 7. 설계 팀장에게 "에어컨 사용량을 줄이려면 이것부터 손보라"고 단 하나만 짚어 주려 합니다. 단위가 제각각인 제원들을 같은 잣대로 비교했을 때 냉방부하에 가장 큰 영향을 주는 변수는 무엇인가요?

- **정답**: `건물조밀도`
- **복습 개념**: 표준화 회귀계수(β)입니다. 원래 계수 B는 변수마다 단위가 달라(건물조밀도는 0.62~0.98, 벽면적은 245~416㎡) 크기를 직접 비교할 수 없습니다. 모든 변수를 평균 0·표준편차 1로 맞춘 뒤의 계수인 β를 비교해야 "누가 더 세게 미는가"를 말할 수 있습니다.
- **풀이 접근**: 최종 모형의 회귀계수표에서 표준화 계수(β) 열을 보고 절댓값이 가장 큰 변수를 찾습니다.
- **근거(계산 결과)**: β는 건물조밀도 **0.740**, 벽면적 0.579, 유리창면적 0.199 입니다. 원래 계수 B만 보면 건물조밀도 67.51 vs 벽면적 41.20 이라 언뜻 비슷해 보이지만, 건물조밀도는 0.1만 움직여도 실제로는 큰 변화라 단위를 맞춰 비교해야 합니다. 세 변수 모두 계수가 양수이므로 조밀도가 높을수록, 벽면적이 넓을수록, 유리창이 클수록 냉방부하가 커집니다.
- **결론**: 당신이 팀장에게 보고할 답은 **건물조밀도**입니다. 중복 16행을 걷어내고(752행), 벽면적을 로그 변환하고, 표면적·지붕면적·높이를 공선성 때문에 덜어낸 뒤, 유의하지 않은 유리창면적분포까지 뺀 최종 3변수 모형은 냉방부하의 77.4%를 설명하며, 그중 가장 강하게 냉방부하를 밀어 올리는 것은 건물조밀도입니다. 다만 네 가지 가정이 모두 깨져 있으므로 "조밀도를 X만큼 낮추면 부하가 정확히 Y만큼 준다"는 식의 수치 약속은 하지 말고, 우선순위를 가리키는 근거로만 제시하는 것이 안전합니다.